# **LLM Meal Planner**

This notebook tests the `LLMMealPlanner`, using a large language model to generate meal plans based on user preferences and nutritional requirements by prompting the model with a structured input format.

Specifically, the model tested is OpenAI's GPT 5.4 Mini.

In [1]:
import os
import sys
from datetime import datetime, timedelta

from openai import OpenAI

sys.path.insert(0, os.path.abspath(".."))

import json
from random import randint, random, seed

from dotenv import load_dotenv

from engines import (
    LLMMealPlanner,
    filter_and_add_recipes,
    get_pantry_ingredient,
    load_all_ingredients,
    make_preferences,
    parse_recipes,
)
from models import (
    MealPlanningEnvironment,
    Pantry,
)

load_dotenv()

True

## Environment Setup

In [ ]:
seed(1)

In [ ]:
preferences = make_preferences()

In [ ]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [ ]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [10]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-05-05
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-08
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-10
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
with open("../recipe_extraction/supplemented_structured_recipes.json", "r") as file:
    all_recipes = json.load(file)

In [12]:
all_recipes = parse_recipes(all_recipes)

In this notebook, the number of filtered recipes is kept small (10) to keep the amount of tokens used low (saving on API costs).

In [ ]:
NUM_FILTERED_RECIPES = 10
NUM_EXTRA_RECIPES = 10

In [14]:
final_recipes = filter_and_add_recipes(
    all_recipes, [ingredient.name for ingredient in pantry_ingredients], NUM_FILTERED_RECIPES, NUM_EXTRA_RECIPES
)

In [15]:
print(f"Number of recipes before filtering: {len(all_recipes)}")
print(f"Number of recipes after filtering: {len(final_recipes)}")

Number of recipes before filtering: 19716
Number of recipes after filtering: 20


In [16]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=final_recipes,
    pantry=pantry,
    preferences=preferences,
)

In [17]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [18]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [19]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

## Running the LLM Meal Planner

In [20]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

planner = LLMMealPlanner(
    meal_planning_environment,
    llm_client=client,
    model_name="gpt-5.4-mini",
    prompt_filepath="./LLMMealPlannerPrompt.txt",
)

In [21]:
print(planner.prompt)

You are a meal planning assistant. Your task is to select a 7-day meal plan (3 meals per day = 21 meals total) from a provided list of recipes, optimised according to the user's preferences and pantry.

You will be given:
1. A numbered list of available recipes (index, name, dietary tags, calories, protein, key ingredients).
2. The user's pantry: ingredients currently in stock with their available quantities (in grams), with days until expiry and estimated financial cost.
3. The user's preferences: daily calorie target, daily protein target, weekly grocery budget, and any dietary restrictions (vegetarian, vegan, gluten-free, lactose-free).

This list of input should always remain untouched. You must not add any new recipes or ingredients, and you must not modify the pantry or user preferences. Your task is solely to select the optimal combination of recipes from the provided list based on the given information.

---

YOUR GOAL

Select exactly 21 recipe indices (integers) from the provi

In [22]:
best_meal_plan, best_fitness_score = planner.generate_meal_plan()

### *LLM Meal Plan Results*

In [23]:
best_meal_plan_recipes = [meal_planning_environment.recipes[i] for i in best_meal_plan]

In [24]:
print([recipe.name for recipe in best_meal_plan_recipes])

['Fried Rice with Ham, Egg, and Scallions', 'Make-Your-Own Salad with Lemon Garlic Dressing', 'Lemon-Herb Mayonnaise', 'Fried Rice with Ham, Egg, and Scallions', 'Vegetable and Wild Rice Salad', 'Basil Garlic Mayonnaise', 'Broccoli with Garlic and Parmesan Cheese', 'Broccoli Rabe and Provolone Grinders', 'Lemon-Herb Mayonnaise', 'Fried Rice with Ham, Egg, and Scallions', 'Celery Root Mashed Potatoes', 'Basil Garlic Mayonnaise', 'Chicken & Broccoli with Crispy Noodles', 'Make-Your-Own Salad with Lemon Garlic Dressing', 'Spiced Hot Chocolate', 'Fried Rice with Ham, Egg, and Scallions', 'Vegetable and Wild Rice Salad', 'Basil Garlic Mayonnaise', 'Broccoli with Garlic and Parmesan Cheese', 'Broccoli Rabe and Provolone Grinders', 'Lemon-Herb Mayonnaise']


In [25]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Meal 1,Meal 2,Meal 3
Day 1,"Fried Rice with Ham, Egg, and Scallions",Make-Your-Own Salad with Lemon Garlic Dressing,Lemon-Herb Mayonnaise
Day 2,"Fried Rice with Ham, Egg, and Scallions",Vegetable and Wild Rice Salad,Basil Garlic Mayonnaise
Day 3,Broccoli with Garlic and Parmesan Cheese,Broccoli Rabe and Provolone Grinders,Lemon-Herb Mayonnaise
Day 4,"Fried Rice with Ham, Egg, and Scallions",Celery Root Mashed Potatoes,Basil Garlic Mayonnaise
Day 5,Chicken & Broccoli with Crispy Noodles,Make-Your-Own Salad with Lemon Garlic Dressing,Spiced Hot Chocolate
Day 6,"Fried Rice with Ham, Egg, and Scallions",Vegetable and Wild Rice Salad,Basil Garlic Mayonnaise
Day 7,Broccoli with Garlic and Parmesan Cheese,Broccoli Rabe and Provolone Grinders,Lemon-Herb Mayonnaise


In [26]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0.000000,800.000000,1d
1,broccoli,1500,683.403794,816.596206,4d
2,rice,1000,316.800000,683.200000,6d


In [27]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 62
Total cost: €18.90


,Ingredient,Quantity to Buy (g),Cost (€)
0,"6""-8""-long French rolls",160.0,0.16
1,Chicken Stock,44.4,0.01
2,Fresno chile,20.0,0.18
3,Ground nutmeg,0.6,0.00
4,Kosher salt,20.0,0.11
...,...,...,...
58,white wine vinegar,14.8,0.06
59,white-wine vinegar,8.4,0.07
60,whole milk,52.6,0.51
61,wild rice,99.0,0.69


In [28]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,1056.8 kcal,17.6 g,-943.2 kcal,-32.4 g
Day 2,978.5 kcal,32.8 g,-1021.5 kcal,-17.2 g
Day 3,1040.6 kcal,26.2 g,-959.4 kcal,-23.8 g
Day 4,915.3 kcal,19.9 g,-1084.7 kcal,-30.1 g
Day 5,1468.2 kcal,71.2 g,-531.8 kcal,21.2 g
Day 6,978.5 kcal,32.8 g,-1021.5 kcal,-17.2 g
Day 7,1040.6 kcal,26.2 g,-959.4 kcal,-23.8 g
